In [83]:
# dataset
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# models
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN, OPTICS, AgglomerativeClustering
from sklearn.metrics.pairwise import rbf_kernel

# evals
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# preprocessing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# training
from sklearn.model_selection import train_test_split

### Method Preparation

In [84]:
def train_model(X_train, y_train, method='rf', **kwargs):

    method = method.lower()

    if method == 'logistic':
        model = LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            **kwargs
        )

    elif method == 'perceptron':
        model = Perceptron(**kwargs)

    elif method == 'svm':
        model = SVC(
            probability=True,
            class_weight='balanced',
            **kwargs
        )

    elif method == 'decision_tree':
        model = DecisionTreeClassifier(**kwargs)

    elif method == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            random_state=42,
            **kwargs
        )

    elif method == 'gradient_boosting':
        model = GradientBoostingClassifier(**kwargs)

    elif method == 'adaboost':
        model = AdaBoostClassifier(
            n_estimators=200,
            random_state=42,
            **kwargs
        )

    else:
        raise ValueError(f"Unexpected Method: {method}")

    model.fit(X_train, y_train)
    return model

In [85]:
def perform_clustering(X, method='kmeans', n_clusters=3, **kwargs):
    
    method = method.lower()

    # ---------- k-Means ----------
    if method == 'kmeans':
        model = KMeans(
            n_clusters=n_clusters,
            init='k-means++',
            random_state=42,
            **kwargs
        )
        labels = model.fit_predict(X)

    # ---------- Kernel k-Means（近似实现） ----------
    elif method == 'kernel_kmeans':
        gamma = kwargs.get('gamma', 1.0)

        K = rbf_kernel(X, gamma=gamma)  # 核矩阵
        model = KMeans(n_clusters=n_clusters, random_state=42)
        labels = model.fit_predict(K)

    # ---------- DBSCAN ----------
    elif method == 'dbscan':
        model = DBSCAN(**kwargs)
        labels = model.fit_predict(X)

    # ---------- OPTICS ----------
    elif method == 'optics':
        model = OPTICS(**kwargs)
        labels = model.fit_predict(X)

    # ---------- Agglomerative ----------
    elif method == 'agglomerative':
        model = AgglomerativeClustering(
            n_clusters=n_clusters,
            **kwargs
        )
        labels = model.fit_predict(X)

    # ---------- GMM ----------
    elif method == 'gmm':
        from sklearn.mixture import GaussianMixture

        model = GaussianMixture(
            n_components=n_clusters,
            random_state=42,
            **kwargs
        )
        model.fit(X)
        labels = model.predict(X)

    else:
        raise ValueError(f"Unexpected Method: {method}")

    return model, labels

In [86]:
def evaluate_classification(model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    classes = np.unique(y_test)
    is_multiclass = len(classes) > 2

    avg = 'macro'
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average=avg, zero_division=0),
        'recall': recall_score(y_test, y_pred, average=avg),
        'f1': f1_score(y_test, y_pred, average=avg),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        if is_multiclass:
            results['roc_auc'] = roc_auc_score(
                y_test, y_proba, multi_class='ovr', average='macro'
            )
        else:
            results['roc_auc'] = roc_auc_score(y_test, y_proba[:, 1])

    return results

In [87]:
def evaluate_clustering(X, labels, y_true=None):

    results = {}

    if len(set(labels)) > 1:
        results['silhouette'] = silhouette_score(X, labels)
        results['calinski_harabasz'] = calinski_harabasz_score(X, labels)
        results['davies_bouldin'] = davies_bouldin_score(X, labels)

    if y_true is not None:
        results['ARI'] = adjusted_rand_score(y_true, labels)
        results['NMI'] = normalized_mutual_info_score(y_true, labels)

    return results

### Dataset

In [88]:
file_path = 'data/data.csv'
df = pd.read_csv(file_path, sep=';')

In [89]:
display(df.head())

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nationality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [90]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nationality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

In [91]:
print("--- Missing Values Check ---")
missing_values = df.isnull().sum()
columns_with_missing = missing_values[missing_values > 0]
if not columns_with_missing.empty:
    print("Columns with missing values and their counts:")
    print(columns_with_missing)
else:
    print("Great news! There are NO missing values in the entire dataset.")
total_missing = df.isnull().sum().sum()
print(f"Total missing values in the dataset: {total_missing}")

--- Missing Values Check ---
Great news! There are NO missing values in the entire dataset.
Total missing values in the dataset: 0


In [92]:
X_raw = df.drop(columns=['Target'])

In [93]:
binary_cols = [
    'Daytime/evening attendance',
    'Displaced',
    'Educational special needs',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    'International',
]

cat_cols = [
    'Marital status',
    'Application mode',
    'Course',
    'Previous qualification',
    'Nationality',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

num_cols = [c for c in X_raw.columns if c not in binary_cols + cat_cols]

In [94]:
print(len(X_raw.columns) == len(binary_cols) + len(cat_cols) + len(num_cols))

True


### Features

In [95]:
X = X_raw.copy()

In [96]:
X['grade_trend'] = (
    X['Curricular units 2nd sem (grade)']
    - X['Curricular units 1st sem (grade)']
)
X['workload_change'] = (
    (X['Curricular units 2nd sem (enrolled)'] +
     X['Curricular units 2nd sem (evaluations)'])
    -
    (X['Curricular units 1st sem (enrolled)'] +
     X['Curricular units 1st sem (evaluations)'])
)
X['fail_rate_2nd'] = (
    (X['Curricular units 2nd sem (enrolled)'] -
     X['Curricular units 2nd sem (approved)']) /
    (X['Curricular units 2nd sem (enrolled)'] + 1)
)
X['grade_per_eval_2nd'] = (
    X['Curricular units 2nd sem (grade)'] /
    (X['Curricular units 2nd sem (evaluations)'] + 1)
)
X['approval_rate_1st'] = (
    X['Curricular units 1st sem (approved)']
    / (X['Curricular units 1st sem (evaluations)'] + 1)
)
X['approval_rate_2nd'] = (
    X['Curricular units 2nd sem (approved)']
    / (X['Curricular units 2nd sem (evaluations)'] + 1)
)

num_cols += [
    'grade_trend',
    'workload_change',
    'fail_rate_2nd',
    'grade_per_eval_2nd',
    'approval_rate_1st',
    'approval_rate_2nd',
]

In [97]:
# Near-zero-variance detection
NZV_THRESHOLD = 0.95
nzv_cols = []
for col in X.columns:
    top_pct = X[col].value_counts(normalize=True).iloc[0]
    if top_pct > NZV_THRESHOLD:
        nzv_cols.append((col, top_pct))
        print(f'  NZV → {col}: dominant value occupies {top_pct:.1%}')

# High-correlation detection
CORR_THRESHOLD = 0.90
corr_matrix = X.corr().abs()
high_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if r > CORR_THRESHOLD:
            high_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], r))
            print(f'  CORR → {corr_matrix.columns[i]}  ↔  {corr_matrix.columns[j]}  (r={r:.3f})')


  NZV → Nationality: dominant value occupies 97.5%
  NZV → Educational special needs: dominant value occupies 98.8%
  NZV → International: dominant value occupies 97.5%
  CORR → Mother's occupation  ↔  Father's occupation  (r=0.910)
  CORR → Curricular units 1st sem (credited)  ↔  Curricular units 2nd sem (credited)  (r=0.945)
  CORR → Curricular units 1st sem (enrolled)  ↔  Curricular units 2nd sem (enrolled)  (r=0.943)
  CORR → Curricular units 1st sem (approved)  ↔  Curricular units 2nd sem (approved)  (r=0.904)


In [98]:
drop_nzv = [
    'Nationality',
    'Educational special needs',
    'International',
]
X.drop(columns=drop_nzv, inplace=True)
drop_corr = [
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    "Mother's occupation",
]
X.drop(columns=drop_corr, inplace=True)

In [99]:
cat_cols = [c for c in cat_cols if c not in drop_nzv + drop_corr]
num_cols = [c for c in num_cols if c not in drop_nzv + drop_corr]
binary_cols = [c for c in binary_cols if c not in drop_nzv + drop_corr]

In [100]:


num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
])

binary_transformer = 'passthrough'

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
        ('bin', binary_transformer, binary_cols),
    ],
    remainder='passthrough',
)

X_processed = preprocessor.fit_transform(X)
X_dense = X_processed if isinstance(X_processed, pd.DataFrame) else pd.DataFrame(X_processed)

In [101]:
print(f'Original features:         {X_raw.shape[1]}')
print(f'After add & filtering:     {X.shape[1]}')
print(f'After transformation:      {X_dense.shape[1]}')
print(f'   Numeric (StandardScaler):   {len(num_cols)}')
print(f'   Nominal (OneHotEncoder):    {len(cat_cols)} raw -> {X_dense.shape[1] - len(num_cols) - len(binary_cols)} encoded columns')
print(f'   Binary  (passthrough):      {len(binary_cols)}')

Original features:         36
After add & filtering:     36
After transformation:      189
   Numeric (StandardScaler):   23
   Nominal (OneHotEncoder):    7 raw -> 160 encoded columns
   Binary  (passthrough):      6


In [102]:
PCA_dim = min(X_dense.shape[1], 50)
X_pca = PCA(n_components=PCA_dim, random_state=42).fit_transform(X_processed)

In [103]:
# tsne = TSNE(
#     n_components=2,
#     random_state=42,
#     init="pca",
#     learning_rate="auto",
#     perplexity=30,
# )
# X_tsne = tsne.fit_transform(X_pca)

# tsne_df = pd.DataFrame(X_tsne, columns=["tsne1", "tsne2"])

### 3 Classes

In [104]:
y_3classes = df['Target'].copy()
print(f'Target distribution:')
for cls, cnt in y_3classes.value_counts().items():
    print(f'  {cls:12s}: {cnt:5d}  ({cnt / len(y_3classes) * 100:.1f} %)')

Target distribution:
  Graduate    :  2209  (49.9 %)
  Dropout     :  1421  (32.1 %)
  Enrolled    :   794  (17.9 %)


In [105]:
# plot_df = tsne_df.copy()
# plot_df["label"] = y_3classes.values

# plt.figure(figsize=(8, 6))
# sns.scatterplot(
#     data=plot_df,
#     x="tsne1",
#     y="tsne2",
#     hue="label",
#     palette="tab10",
#     s=25,
#     alpha=0.8,
# )
# plt.title("t-SNE (3 classes)")
# plt.legend(title="Target", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.show()

In [106]:
cluster_methods = {
    'kmeans': {},
    'kernel_kmeans': {},
    'agglomerative': {'linkage': 'ward'},
    'gmm': {},
}

cluster_rows = []
for method, kwargs in cluster_methods.items():
    model, labels = perform_clustering(
        X_pca, method=method, n_clusters=3, **kwargs
)
    labels = np.asarray(labels)
    effective_k = len(set(labels)) - (1 if -1 in labels else 0)
    noise_rate = float(np.mean(labels == -1)) if -1 in labels else 0.0

    metrics = evaluate_clustering(X_pca, labels, y_true=y_3classes)
    cluster_rows.append({
        'method': method,
        'effective_k': effective_k,
        'noise_rate': noise_rate,
        **metrics,
    })

cluster_results = pd.DataFrame(cluster_rows)
cluster_results = cluster_results.sort_values(
    by=['silhouette', 'ARI', 'NMI'], ascending=[False, False, False]
)
display(cluster_results.round(4))

,method,effective_k,noise_rate,silhouette,calinski_harabasz,davies_bouldin,ARI,NMI
2,agglomerative,3,0.0,0.2732,777.3202,1.4991,0.1439,0.1604
0,kmeans,3,0.0,0.1152,771.9154,2.5841,0.3467,0.2702
3,gmm,3,0.0,0.1014,682.9207,3.2992,0.1826,0.1741
1,kernel_kmeans,3,0.0,-0.1822,19.5426,2.5332,-0.0175,0.0246


In [107]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_3classes, test_size=0.30, random_state=42, stratify=y_3classes
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

model_specs = {
    'LogisticRegression': ('logistic', {}),
    'Perceptron': ('perceptron', {}),
    'SVM': ('svm', {}),
    'DecisionTree': ('decision_tree', {}),
    'RandomForest': ('random_forest', {}),
    'GradientBoosting': ('gradient_boosting', {}),
    'AdaBoost': ('adaboost', {}),
}

classification_rows = []
confusion_mats = {}
for name, (method, kwargs) in model_specs.items():
    model = train_model(X_train, y_train, method=method, **kwargs)
    res = evaluate_classification(model, X_test, y_test)
    confusion_mats[name] = res.pop('confusion_matrix')
    res['model'] = name
    classification_rows.append(res)

classification_results = pd.DataFrame(classification_rows)
classification_results = classification_results.sort_values(
    by=['f1', 'accuracy'], ascending=[False, False]
).reset_index(drop=True)
display(classification_results.round(4))

d:\Workspace\school\DSAA2011\machine_learning_project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,accuracy,precision,recall,f1,roc_auc,model
0,0.7568,0.7246,0.7330,0.7201,0.8857,LogisticRegression
1,0.7726,0.7180,0.7001,0.7068,0.8897,GradientBoosting
2,0.7402,0.7120,0.7187,0.7047,0.8868,SVM
3,0.7779,0.7288,0.6892,0.6999,0.8872,RandomForest
4,0.7628,0.7057,0.6827,0.6907,0.8522,AdaBoost
5,0.7199,0.6808,0.6771,0.6728,NaN,Perceptron
6,0.6672,0.6031,0.6057,0.6038,0.7185,DecisionTree


### 2 Classes

In [108]:
y_2classes = df['Target'].map({'Dropout': 'Dropout', 'Graduate': 'Non-Dropout', 'Enrolled': 'Non-Dropout'})
print(f'Target distribution:')
for cls, cnt in y_2classes.value_counts().items():
    print(f'  {cls:12s}: {cnt:5d}  ({cnt / len(y_2classes) * 100:.1f} %)')

Target distribution:
  Non-Dropout :  3003  (67.9 %)
  Dropout     :  1421  (32.1 %)


In [109]:
# plot_df = tsne_df.copy()
# plot_df["label"] = y_2classes.values

# plt.figure(figsize=(8, 6))
# sns.scatterplot(
#     data=plot_df,
#     x="tsne1",
#     y="tsne2",
#     hue="label",
#     palette="Set2",
#     s=25,
#     alpha=0.8,
# )
# plt.title("t-SNE (2 classes)")
# plt.legend(title="Target", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.show()

In [110]:
cluster_methods_2 = {
    'kmeans': {},
    'kernel_kmeans': {},
    'agglomerative': {'linkage': 'ward'},
    'gmm': {},
}

cluster_rows_2 = []
for method, kwargs in cluster_methods_2.items():
    model, labels = perform_clustering(
        X_pca, method=method, n_clusters=2, **kwargs
)
    labels = np.asarray(labels)
    effective_k = len(set(labels)) - (1 if -1 in labels else 0)
    noise_rate = float(np.mean(labels == -1)) if -1 in labels else 0.0

    metrics = evaluate_clustering(X_pca, labels, y_true=y_2classes)
    cluster_rows_2.append({
        'method': method,
        'effective_k': effective_k,
        'noise_rate': noise_rate,
        **metrics,
    })

cluster_results_2 = pd.DataFrame(cluster_rows_2)
cluster_results_2 = cluster_results_2.sort_values(
    by=['silhouette', 'ARI', 'NMI'], ascending=[False, False, False]
)
display(cluster_results_2.round(4))

,method,effective_k,noise_rate,silhouette,calinski_harabasz,davies_bouldin,ARI,NMI
3,gmm,2,0.0,0.2996,1146.5759,1.5696,0.3635,0.2579
0,kmeans,2,0.0,0.2977,1154.3453,1.5700,0.3712,0.2641
2,agglomerative,2,0.0,0.2843,1063.5230,1.5215,0.3322,0.2370
1,kernel_kmeans,2,0.0,-0.1460,2.5282,1.5555,-0.0012,0.0014


In [111]:
X_train_raw_2, X_test_raw_2, y_train_2, y_test_2 = train_test_split(
    X, y_2classes, test_size=0.30, random_state=42, stratify=y_2classes
)

X_train_2 = preprocessor.fit_transform(X_train_raw_2)
X_test_2 = preprocessor.transform(X_test_raw_2)

model_specs_2 = {
    'LogisticRegression': ('logistic', {}),
    'Perceptron': ('perceptron', {}),
    'SVM': ('svm', {}),
    'DecisionTree': ('decision_tree', {}),
    'RandomForest': ('random_forest', {}),
    'GradientBoosting': ('gradient_boosting', {}),
    'AdaBoost': ('adaboost', {}),
}

classification_rows_2 = []
confusion_mats_2 = {}
for name, (method, kwargs) in model_specs_2.items():
    model = train_model(X_train_2, y_train_2, method=method, **kwargs)
    res = evaluate_classification(model, X_test_2, y_test_2)
    confusion_mats_2[name] = res.pop('confusion_matrix')
    res['model'] = name
    classification_rows_2.append(res)

classification_results_2 = pd.DataFrame(classification_rows_2)
classification_results_2 = classification_results_2.sort_values(
    by=['f1', 'accuracy'], ascending=[False, False]
).reset_index(drop=True)
display(classification_results_2.round(4))

d:\Workspace\school\DSAA2011\machine_learning_project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,accuracy,precision,recall,f1,roc_auc,model
0,0.8788,0.8752,0.8410,0.8549,0.9173,GradientBoosting
1,0.8712,0.8518,0.8540,0.8529,0.9139,SVM
2,0.8660,0.8441,0.8538,0.8486,0.9189,LogisticRegression
3,0.8705,0.8593,0.8380,0.8472,0.9135,AdaBoost
4,0.8667,0.8595,0.8279,0.8407,0.9117,RandomForest
5,0.8456,0.8211,0.8339,0.8267,NaN,Perceptron
6,0.8012,0.7726,0.7685,0.7705,0.7685,DecisionTree
